In [37]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html

In [ ]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

In [2]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [3]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

70

In [4]:
UNIVERSE = {}

for ticker in INDEX_DATA:
    if datetime.datetime(2025, 2, 18) in INDEX_DATA[ticker].index:
        UNIVERSE[ticker] = INDEX_DATA[ticker].copy(deep = True)

In [5]:
print(f"Assets under Analysis: {len(UNIVERSE)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 67
Total Assets: 70


## Z Score Calculation

In [6]:
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
target_date = datetime.datetime(2025, 2, 18)
years = 4
duration = years * 250

for ticker in UNIVERSE:
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [29]:
display_number = 20

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'PE_TTM'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'DIVIDENDYIELD'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='PE_TTM', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='DIVIDENDYIELD', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number)
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number)

top_Z_PE_TTM.loc[:, "Name"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "Name", "PE_TTM"]]

top_Z_DIVIDENDYIELD.loc[:, "Name"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "Name", "DIVIDENDYIELD"]]

C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_56016\2912958680.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_Z_PE_TTM.loc[:, "Name"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_56016\2912958680.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_Z_DIVIDENDYIELD.loc[:, "Name"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)


In [45]:
INDEX_DATA['000300.SH'].index[-1].strftime("%Y-%m-%d"), f"{years} years"

('2025-02-19', '4 years')

In [43]:
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)

,Ticker,Name,PE_TTM
31,707717L.MI,MSCI中国A股质量,-0.924304
11,601688.SH,华泰证券,-0.852808
15,H30533.CSI,中概互联网,-0.852759
2,000913.SH,医药,-0.837543
36,HSCGSI.HI,恒生消费,-0.781023
3,399814.SZ,农业,-0.668809
7,HSTECH.HI,恒生科技,-0.665321
29,HSSCHI.HI,恒生港股通医疗,-0.641141
16,HSIII.HI,恒生互联网,-0.598561
43,HSHK35.HI,恒生香港35,-0.590067
